# Bias Mitigation — Adversarial Debiasing (in-processing)

**AIF360 class:** `aif360.algorithms.inprocessing.AdversarialDebiasing` &nbsp;|&nbsp; requires `pip install -e .[mitigation,deep]` (TensorFlow)

Trains a predictor against an adversary that tries to recover the protected attribute, discouraging the predictor from encoding it.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression

from fair_ed.mitigation.base import (
    load_splits, group_dicts, to_aif360_dataset,
    dataset_from_predictions, evaluate_subgroups,
)
from fair_ed.modeling import prepare_xy, standardize, FEATURE_COLUMNS

# --- Prediction task and binary protected attribute (edit to taste) ---
OUTCOME = "outcome_hospitalization"   # e.g. outcome_critical, outcome_sepsis, outcome_pe, ...
PROTECTED = "sex_priv"                 # 1 = male (privileged), 0 = female (unprivileged)

df_train, df_test = load_splits()
for _df in (df_train, df_test):
    _df[PROTECTED] = (_df["gender"].astype(str).str.upper().str[0] == "M").astype(int)

X_train, X_test, y_train, y_test = prepare_xy(df_train, df_test, OUTCOME)
X_train, X_test, _ = standardize(X_train, X_test)
FEAT = list(X_train.columns)

train_frame = X_train.assign(**{PROTECTED: df_train[PROTECTED].to_numpy(), OUTCOME: y_train.to_numpy()})
test_frame = X_test.assign(**{PROTECTED: df_test[PROTECTED].to_numpy(), OUTCOME: y_test.to_numpy()})

priv, unpriv = group_dicts(PROTECTED)

def subgroup_labels(ds):
    return np.where(ds.protected_attributes.ravel() == 1, "Male", "Female")

## Apply Adversarial Debiasing and evaluate subgroups

In [ ]:
from fair_ed.mitigation.inprocessing import adversarial_debiasing

train_ds = to_aif360_dataset(train_frame, FEAT, OUTCOME, PROTECTED)
test_ds = to_aif360_dataset(test_frame, FEAT, OUTCOME, PROTECTED)

# Adversarial Debiasing trains under a TensorFlow v1 session (requires the
# `deep` extra). Returns predicted labels and scores on the test set.
pred_ds = adversarial_debiasing(train_ds, test_ds, priv, unpriv, num_epochs=50)

y_pred = pred_ds.labels.ravel()
y_score = pred_ds.scores.ravel()

ci = evaluate_subgroups(test_ds.labels.ravel(), y_pred, y_score, subgroup_labels(test_ds), B=200)
print(ci.pivot(index="Group", columns="Metric", values="Mean").round(3))